In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Sensor calibration

**New concept: `direction='nearest'` in `merge_asof`**

You've used `direction='backward'` (most recent past) and `direction='forward'` (next future). `direction='nearest'` picks whichever is closer — past or future:

```python
pd.merge_asof(left, right, on='time', direction='nearest')
```

If past and future are equidistant, pandas takes the past value.

A quality control sensor is calibrated at fixed times during the day. Readings happen at irregular times between calibrations. For each reading, attach the *nearest* calibration — it might be slightly before or slightly after.

1. Sort both by `time` and merge with `direction='nearest'`.
2. Add `adjusted = value * factor`. Which reading had the highest adjusted value? Use `np.argmax`.
3. Named-agg groupby by `station`: mean `adjusted` and mean `factor`. Which station's readings required more correction on average?

In [8]:
calibrations = pd.DataFrame({
    'time':   pd.to_datetime(['08:00','10:30','13:00','15:30'], format='%H:%M'),
    'factor': [1.02, 0.98, 1.01, 0.99],
})

readings = pd.DataFrame({
    'time':    pd.to_datetime(['07:45','09:00','11:30','12:00','14:20','16:00'], format='%H:%M'),
    'value':   [45.2, 55.0, 48.9, 52.3, 49.7, 47.1],
    'station': ['A', 'B', 'A', 'B', 'A', 'B'],
})

# Your code here
calibrations = calibrations.sort_values('time')
readings = readings.sort_values('time')


m = pd.merge_asof(
    readings, 
    calibrations, 
    on = 'time',
    direction= 'nearest'
)

m['adjusted'] = m['value']*m['factor']

print(m.iloc[np.argmax(m['adjusted'])],'had the highest adjusted value')


g = m.groupby("station").agg(
    ma = ('adjusted','mean'),
    mf = ('factor','mean')
)

print(g['ma'].idxmax(),'requres more')

time        1900-01-01 09:00:00
value                      55.0
station                       B
factor                     1.02
adjusted                   56.1
Name: 1, dtype: object had the highest adjusted value
B requres more


---

## Level 2 — Product orders with stale price guard

You've used `by=` and `tolerance` separately. Here they combine: for each order, attach the most recent price for **that product**, but only if the price was updated within the last 30 days. A price older than 30 days is considered stale — return NaN instead.

```python
pd.merge_asof(orders, prices, on='date', by='product', tolerance=pd.Timedelta('30D'))
```

Sort both DataFrames by `['product', 'date']` before merging.

1. Merge as described. How many orders received no valid price? List them by product.
2. Add `revenue = qty * price`. Use `np.nanmean` to find the average revenue per order across all orders.
3. For matched orders only (drop NaN rows): named-agg groupby by `product` — total revenue and mean `qty`.

In [26]:
prices = pd.DataFrame({
    'date':    pd.to_datetime(['2023-01-05','2023-02-10','2023-04-15',
                               '2023-01-20','2023-03-01','2023-05-20']),
    'product': ['P1','P1','P1','P2','P2','P2'],
    'price':   [100, 105, 98, 200, 210, 195],
})

orders = pd.DataFrame({
    'date':    pd.to_datetime(['2023-01-25','2023-02-05','2023-04-10','2023-04-20',
                               '2023-02-15','2023-03-15','2023-04-01','2023-06-10']),
    'product': ['P1','P1','P1','P1','P2','P2','P2','P2'],
    'qty':     [10, 5, 8, 12, 6, 4, 9, 7],
})

# Your code here

prices = prices.sort_values('date')
orders = orders.sort_values('date')

m = pd.merge_asof(
    orders, 
    prices, 
    by = 'product',
    on = 'date',
    tolerance = pd.Timedelta('30D')
)


print(m.groupby('product')['price'].apply(lambda x:  x.isna().sum()))

m['revenue'] = m['qty'] *m['price']
print('average revenue is: ', np.nanmean(m['revenue']))
mn = m.dropna()

g = mn.groupby('product').agg(
    tr = ('revenue','sum'),
    mq = ('qty','mean')
)

print(g)

product
P1    2
P2    1
Name: price, dtype: int64
average revenue is:  1116.2
             tr         mq
product                   
P1       2176.0  11.000000
P2       3405.0   5.666667


---

## Level 3 — Sales vs marketing spend

`sales` and `marketing` report figures on alternating weeks — neither has a complete record.

Build a full weekly view, compute ROI, and find when the marketing was most efficient.

1. `merge_ordered` with `fill_method='ffill'`, drop NaN rows.
2. Add `roi = revenue / spend`. Which week had the best ROI? Use `.idxmax()` then `.loc[]`.
3. `np.corrcoef` on `revenue` and `spend` — does higher spend correlate with higher revenue?
4. `np.percentile` on `roi` — find the 25th and 75th percentiles.
5. List comprehension with `zip`: collect weeks where `roi > 6.0`.

In [38]:
sales = pd.DataFrame({
    'week':    pd.to_datetime(['2023-01-02','2023-01-16','2023-01-30','2023-02-13','2023-02-27']),
    'revenue': [45000, 52000, 48000, 61000, 55000],
})

marketing = pd.DataFrame({
    'week':  pd.to_datetime(['2023-01-09','2023-01-23','2023-02-06','2023-02-20','2023-03-06']),
    'spend': [8000, 9500, 7500, 11000, 10000],
})

# Your code here


m = pd.merge_ordered(sales, marketing, on = 'week', fill_method='ffill').dropna()
m['roi'] = m['revenue']/m['spend']

print(m.loc[m['roi'].idxmax(),'week'],'had the best ROI')

cr = np.corrcoef(m['revenue'],m['spend'])[0,1]
print(f'correlation is {cr:.2f}, slightly correlated')

rr = np.percentile(m['roi'],q = [25, 75])
print(rr)

[w for w, r in zip(m['week'],m['roi']) if r>6.0]


2023-02-13 00:00:00 had the best ROI
correlation is 0.37, slightly correlated
[5.47368421 6.4       ]


[Timestamp('2023-01-16 00:00:00'),
 Timestamp('2023-02-06 00:00:00'),
 Timestamp('2023-02-13 00:00:00')]